# 05 — RoBERTa Sentiment Analysis

**CardiffNLP Twitter-RoBERTa** is a transformer model pre-trained on 58M tweets. It understands context and narrative tone — not just surface vocabulary. For war journalism, this means it reads factual reporting as *neutral*, missing the raw language signals VADER catches.

| Section | Content |
|---|---|
| 1 | Load pre-scored results |
| 2 | RoBERTa distribution vs VADER |
| 3 | Model agreement analysis |
| 4 | VADER vs RoBERTa: the key distinction |

> **Next step:** `06_Crude_Oil_Correlation.ipynb`

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
from src.config import SENTIMENT_RESULTS_FILE

# Load pre-scored results (from main.py run)
sent_df = pd.read_csv(SENTIMENT_RESULTS_FILE)
print(f'Scored articles: {len(sent_df)}')
print(f'Columns: {list(sent_df.columns)}')

## 1. Apply RoBERTa (or use cached results)

RoBERTa downloads ~500MB model on first run, then caches locally in `models/roberta_sentiment/`.

In [ ]:
# To re-run RoBERTa inference:
# from src.sentiment import apply_roberta
# sent_df = apply_roberta(sent_df)

# Using cached results from main.py:
if 'roberta_label' in sent_df.columns:
    print('RoBERTa results loaded from cache.')
    roberta_counts = sent_df['roberta_label'].value_counts()
    total = len(sent_df)
    print('\nRoBERTa Distribution:')
    for label, n in roberta_counts.items():
        print(f'  {label:10s}: {n:4d}  ({100*n/total:.1f}%)')
else:
    print('RoBERTa columns not found. Run main.py first.')

## 2. VADER vs RoBERTa Distribution Comparison

In [ ]:
from src.visualize import plot_sentiment_distribution

plot_sentiment_distribution(sent_df)
print('Saved: images/07_sentiment_distribution.png')

print('\nSide-by-side comparison:')
comparison = pd.DataFrame({
    'VADER': sent_df['vader_label'].value_counts(),
    'RoBERTa': sent_df['roberta_label'].value_counts() if 'roberta_label' in sent_df.columns else {},
}).reindex(['Positive', 'Neutral', 'Negative'], fill_value=0)
print(comparison.to_string())

## 3. Model Agreement Analysis

How often do VADER and RoBERTa agree on the same article?

In [ ]:
from src.visualize import plot_model_agreement

if 'roberta_label' in sent_df.columns:
    agree_rate = (sent_df['vader_label'] == sent_df['roberta_label']).mean() * 100
    print(f'Overall agreement rate: {agree_rate:.1f}%')

    print('\nCross-tabulation (VADER rows x RoBERTa cols):')
    ct = pd.crosstab(sent_df['vader_label'], sent_df['roberta_label'])
    print(ct.to_string())

    plot_model_agreement(sent_df)
    print('\nSaved: images/12_model_agreement.png')

## 4. The Key Distinction

> **VADER reacts to vocabulary. RoBERTa understands context.**

Most war journalism maintains a factual, neutral tone despite severe subject matter. RoBERTa classifies this correctly as neutral — but misses the vocabulary signals that commodity traders and market-moving algorithms respond to.

In [ ]:
# Sample articles where models disagree
if 'roberta_label' in sent_df.columns:
    disagreed = sent_df[
        (sent_df['vader_label'] == 'Negative') &
        (sent_df['roberta_label'] == 'Neutral')
    ][['title', 'vader_compound', 'roberta_compound', 'vader_label', 'roberta_label']]

    print(f'Articles VADER=Negative but RoBERTa=Neutral: {len(disagreed)}')
    print('\nSamples:')
    for _, r in disagreed.head(5).iterrows():
        print(f'  VADER: {r["vader_compound"]:.3f} | RoBERTa: {r["roberta_compound"]:.3f}')
        print(f'  {r["title"][:100]}')
        print()

## Key Findings

| Metric | VADER | RoBERTa |
|---|---|---|
| Negative | 60.8% | 34.4% |
| Neutral | 20.7% | 64.3% |
| Positive | 18.5% | 1.3% |

RoBERTa's 64.3% Neutral classification reflects correct contextual reading of journalistic prose — but for commodity price correlation, vocabulary-level signals are what matter.

> **Next step:** `06_Crude_Oil_Correlation.ipynb`